In [2]:
!pip install ragas datasets langsmith -q

In [3]:
import json, os, sys, asyncio
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

EVAL_FILE = ROOT / "eval" / "golden_dataset" / "docs" / "eval_v1.jsonl"
eval_set  = [json.loads(l) for l in EVAL_FILE.read_text().splitlines() if l.strip()]
print(f"Eval set: {len(eval_set)} Q&A pairs")

Eval set: 78 Q&A pairs


In [4]:
import asyncio, asyncpg, hashlib, os
from pathlib import Path
from uuid import UUID

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data" / "raw"

from backend.config import settings

EVAL_TENANT_ID  = "00000000-0000-0000-0000-000000000001" 
EVAL_API_KEY    = "ragas-eval-key"
EVAL_API_HASH   = hashlib.sha256(EVAL_API_KEY.encode()).hexdigest()

async def ensure_tenant(pool):
    """Insert eval tenant if it doesn't exist yet."""
    async with pool.acquire() as conn:
        existing = await conn.fetchrow(
            "SELECT tenant_id FROM tenants WHERE tenant_id = $1",
            EVAL_TENANT_ID,
        )
        if not existing:
            await conn.execute(
                "INSERT INTO tenants (tenant_id, name, api_key_hash) VALUES ($1, $2, $3)",
                EVAL_TENANT_ID, "RAGAS Eval Tenant", EVAL_API_HASH,
            )
            print(f"Created tenant {EVAL_TENANT_ID}")
        else:
            print(f"Tenant already exists: {EVAL_TENANT_ID}")

pool = await asyncpg.create_pool(settings.postgres_url.replace("+asyncpg", ""))
await ensure_tenant(pool)

Tenant already exists: 00000000-0000-0000-0000-000000000001


In [5]:
import asyncpg
from pathlib import Path
from backend.config import settings
from backend.connectors.chunkers.heading_aware_chunker import HeadingAwareChunker
from backend.models import Chunk, SourceType
from backend.strategies.embedding.openai_embedding import OpenAIEmbedding
from backend.strategies.vectordb.qdrant_db import QdrantDB

DATA_DIR       = ROOT / "data" / "raw"
EVAL_TENANT_ID = "00000000-0000-0000-0000-000000000001" 

async def ingest_local_docs(tenant_id: str, data_dir: Path):
    chunker  = HeadingAwareChunker()
    embedder = OpenAIEmbedding()
    db       = QdrantDB()

    if not await db.collection_exists(tenant_id):
        await db.create_collection(tenant_id)
        print(f"Created collection: tenant_{tenant_id}")

    files = list(data_dir.rglob("*.md")) + list(data_dir.rglob("*.mdx"))
    print(f"Found {len(files)} files")

    for path in files:
        text = path.read_text(encoding="utf-8")
        source_url = str(path.relative_to(data_dir))   

        metadata = {
            "tenant_id": tenant_id,
            "source_url": source_url,
            "source_type": SourceType.DOCS_SITE.value,
        }
        raw_chunks = chunker.chunk(text, metadata)

        texts = [c.content for c in raw_chunks]
        vectors = await embedder.embed(texts)

        for chunk, vec in zip(raw_chunks, vectors):
            chunk.tenant_id   = tenant_id
            chunk.source_url  = source_url
            chunk.dense_vector = vec

        await db.upsert(raw_chunks)
        print(f"  ✓ {path.name}  ({len(raw_chunks)} chunks)")

await ingest_local_docs(EVAL_TENANT_ID, DATA_DIR)

Found 9 files
  ✓ background-tasks.md  (8 chunks)
  ✓ concepts.md  (28 chunks)
  ✓ cors.md  (8 chunks)
  ✓ bigger-applications.md  (21 chunks)
  ✓ advanced-guide.mdx  (9 chunks)
  ✓ auth-notion.mdx  (7 chunks)
  ✓ auth-google.mdx  (15 chunks)
  ✓ token-security.mdx  (20 chunks)
  ✓ auth-linkedin.mdx  (11 chunks)


In [7]:
import asyncpg
from backend.config import settings
from backend.core.query_pipeline import QueryPipeline
from backend.strategies.embedding.openai_embedding import OpenAIEmbedding
from backend.strategies.embedding.tf_sparse_encoder import TFSparseEncoder
from backend.strategies.llm.openai_llm import OpenAILLM
from backend.strategies.reranker.cohere_reranker import CohereReranker
from backend.strategies.vectordb.qdrant_db import QdrantDB
from backend.strategies.cache.redis_cache import RedisCache
from backend.repositories.postgres_conversation_repo import PostgresConversationRepository

async def build_pipeline():
    pool = await asyncpg.create_pool(settings.postgres_url.replace("+asyncpg", ""))
    cache = RedisCache()
    return QueryPipeline(
        llm=OpenAILLM(),
        embedder=OpenAIEmbedding(),
        sparse_encoder=TFSparseEncoder(),
        vector_db=QdrantDB(),
        reranker=CohereReranker(),
        cache=cache,
        conversation_repo=PostgresConversationRepository(pool),
        observers=[],   # no observers in eval
    )

pipeline = await build_pipeline()
print("Pipeline ready.")


Pipeline ready.


In [8]:
from uuid import uuid4
from tqdm.asyncio import tqdm_asyncio

async def run_eval(pipeline, eval_set):
    records = []
    for qa in eval_set:
        try:
            result = await pipeline.handle(
                query=qa["question"],
                tenant_id=EVAL_TENANT_ID,
                conversation_id=uuid4(),
            )
            await asyncio.sleep(6)  # to avoid hitting cohere rerank rate limit
            records.append({
                "question":          qa["question"],
                "answer":            result.answer,
                "ground_truth":      qa["answer"],
                "contexts":          [c.content for c in result.source_chunks],
            })
        except Exception as e:
            print(f"  [SKIP] {qa['question'][:60]}: {e}")
    return records

records = await run_eval(pipeline, eval_set)
print(f"Collected {len(records)} answers")

Collected 78 answers


In [9]:
import warnings
warnings.filterwarnings("ignore")

from datasets import Dataset
from ragas import aevaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

judge_llm   = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
judge_embed = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

dataset = Dataset.from_list(records)

scores = await aevaluate(
    dataset=dataset,
    metrics=[Faithfulness(), AnswerRelevancy(), ContextPrecision(), ContextRecall()],
    llm=judge_llm,
    embeddings=judge_embed,
)

print(scores)


Evaluating:   0%|          | 0/312 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

{'faithfulness': 0.9083, 'answer_relevancy': 0.8322, 'context_precision': 0.8916, 'context_recall': 0.9487}


In [11]:
import pandas as pd

results_df = scores.to_pandas()

print("Columns:", results_df.columns.tolist())

question_col = next(
    (c for c in ["question", "user_input", "input"] if c in results_df.columns),
    results_df.columns[0]   # fallback to first column
)

metric_cols = [c for c in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
               if c in results_df.columns]

print("\n=== Per-question scores (sample) ===")
print(results_df[[question_col] + metric_cols].head(10).to_string())

print("\n=== Aggregate scores ===")
agg = {col: results_df[col].mean() for col in metric_cols}
for metric, val in agg.items():
    print(f"  {metric:22s}  {val:.3f}")

Columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']

=== Per-question scores (sample) ===
                                                                              user_input  faithfulness  answer_relevancy  context_precision  context_recall
0                     What is one example of a background task mentioned in the passage?      1.000000          0.983217               0.70             1.0
1                                        Why might background tasks be used in a system?      1.000000          0.929668               1.00             1.0
2                How do you import and use BackgroundTasks in a path operation function?      0.750000          0.821761               1.00             1.0
3  What does FastAPI do with the BackgroundTasks parameter in a path operation function?      1.000000          0.967220               0.95             1.0
4                   What types of fu